In [7]:
import tensorflow as tf
import pandas as pd
import numpy as np

In [8]:
# constants

FILE_PATH = "../tf_preprocess/CIDDS-001-external-week1.csv"

In [9]:
data = pd.read_csv(FILE_PATH)

In [10]:
data.columns

Index(['Date first seen', 'Duration', 'Proto', 'Src IP Addr', 'Src Pt',
       'Dst IP Addr', 'Dst Pt', 'Packets', 'Bytes', 'Flows', 'Flags', 'Tos',
       'class', 'attackType', 'attackID', 'attackDescription'],
      dtype='object')

In [11]:
data.head()

,Date first seen,Duration,Proto,Src IP Addr,Src Pt,Dst IP Addr,Dst Pt,Packets,Bytes,Flows,Flags,Tos,class,attackType,attackID,attackDescription
0,2017-03-14 17:43:57.172,81412.697,TCP,EXT_SERVER,8082,OPENSTACK_NET,56978.0,3057,2.1 M,1,.AP...,0,normal,---,---,---
1,2017-03-14 17:43:57.172,81412.697,TCP,OPENSTACK_NET,56978,EXT_SERVER,8082.0,4748,2.5 M,1,.AP...,0,normal,---,---,---
2,2017-03-14 17:43:26.135,81504.787,TCP,EXT_SERVER,8082,OPENSTACK_NET,56979.0,8639,9.1 M,1,.AP...,0,normal,---,---,---
3,2017-03-14 17:43:26.135,81504.787,TCP,OPENSTACK_NET,56979,EXT_SERVER,8082.0,12024,10.3 M,1,.AP...,0,normal,---,---,---
4,2017-03-14 18:17:09.005,82100.692,TCP,EXT_SERVER,8082,OPENSTACK_NET,51649.0,11012,27.2 M,1,.AP.S.,0,normal,---,---,---


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172838 entries, 0 to 172837
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Date first seen    172838 non-null  object 
 1   Duration           172838 non-null  float64
 2   Proto              172838 non-null  object 
 3   Src IP Addr        172838 non-null  object 
 4   Src Pt             172838 non-null  int64  
 5   Dst IP Addr        172838 non-null  object 
 6   Dst Pt             172838 non-null  float64
 7   Packets            172838 non-null  int64  
 8   Bytes              172838 non-null  object 
 9   Flows              172838 non-null  int64  
 10  Flags              172838 non-null  object 
 11  Tos                172838 non-null  int64  
 12  class              172838 non-null  object 
 13  attackType         172838 non-null  object 
 14  attackID           172838 non-null  object 
 15  attackDescription  172838 non-null  object 
dtypes:

In [175]:
data.isnull().sum()

Date first seen      0
Duration             0
Proto                0
Src IP Addr          0
Src Pt               0
Dst IP Addr          0
Dst Pt               0
Packets              0
Bytes                0
Flows                0
Flags                0
Tos                  0
class                0
attackType           0
attackID             0
attackDescription    0
dtype: int64

In [176]:
data["class"].value_counts(normalize=True)

class
suspicious    0.621067
normal        0.287009
unknown       0.091924
Name: proportion, dtype: float64

In [114]:
proto_unique = data["Proto"].unique()
print("Proto - Valores Unicos:", proto_unique)
print("Proto - Valores Unicos (tamanho):", len(proto_unique))

Proto - Valores Unicos: ['TCP  ' 'UDP  ' 'ICMP ' 'GRE  ']
Proto - Valores Unicos (tamanho): 4


In [115]:
flags_unique = data["Flags"].unique()
print("Flags - Valores Unicos:", flags_unique)
print("Flags - Valores Unicos (tamanho):", len(flags_unique))

Flags - Valores Unicos: ['.AP...' '.AP.S.' '....S.' '.A.R..' '.APRS.' '.APRSF' '.AP.SF' '......'
 '  0xdb' '...RS.' '.A..S.' '.A..SF' '.A.RS.' '.A.RSF' '...R..' '.A....'
 '  0xd2' '.A.R.F' '  0xc2' '  0xda' '  0xd7' '  0x53' '  0xdf' '  0xd6'
 '  0xd3']
Flags - Valores Unicos (tamanho): 25


In [116]:
attackType = data["attackType"].unique()
print("attackType - Valores Unicos:", attackType)
print("attackType - Valores Unicos (tamanho):", len(attackType))

attackType - Valores Unicos: ['---']
attackType - Valores Unicos (tamanho): 1


In [12]:
data = data.drop(columns=[
                        "Date first seen",
                        "Src IP Addr",
                        "Dst IP Addr",
                        "attackType",
                        "attackID",
                        "attackDescription"
                        ])

In [178]:
data.info() # depois de remover colunas desnecessárias para o modelo...

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172838 entries, 0 to 172837
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Duration  172838 non-null  float64
 1   Proto     172838 non-null  object 
 2   Src Pt    172838 non-null  int64  
 3   Dst Pt    172838 non-null  float64
 4   Packets   172838 non-null  int64  
 5   Bytes     172838 non-null  object 
 6   Flows     172838 non-null  int64  
 7   Flags     172838 non-null  object 
 8   Tos       172838 non-null  int64  
 9   class     172838 non-null  object 
dtypes: float64(2), int64(4), object(4)
memory usage: 13.2+ MB


In [121]:
data["Proto"].value_counts(normalize=True)

Proto
TCP      0.976637
UDP      0.012920
ICMP     0.010229
GRE      0.000214
Name: proportion, dtype: float64

In [122]:
data["Flags"].value_counts(normalize=True)

Flags
.AP.SF    0.600152
....S.    0.104288
.APRSF    0.083645
.A.R..    0.063059
.A..S.    0.054895
.AP.S.    0.027847
......    0.023363
.APRS.    0.015448
...RS.    0.011161
.A..SF    0.004455
  0xdb    0.003558
.A.RSF    0.003055
.A.RS.    0.002285
...R..    0.001811
  0xc2    0.000417
.A....    0.000208
.A.R.F    0.000139
  0xd7    0.000098
.AP...    0.000046
  0xda    0.000023
  0xdf    0.000017
  0xd2    0.000012
  0x53    0.000006
  0xd6    0.000006
  0xd3    0.000006
Name: proportion, dtype: float64

In [13]:
data["Bytes"].sample(5) # tratar para float32 ou float64

131515        3249
38916          579
27384           46
140901         515
132132         556
Name: Bytes, dtype: object

In [14]:
data["numBytes"] = data["Bytes"].str.split().str[0].astype(float)

In [15]:
data[["numBytes"]].sample(5)

,numBytes
167235,579.0
55714,4045.0
84900,2215.0
164388,1937.0
37978,556.0


In [16]:
data = pd.get_dummies(
    data,
    columns=["Proto", "Flags", "class"],
    dtype=int
)

In [17]:
data = data.drop(columns="Bytes")

In [184]:
data.sample(5)

,Duration,Src Pt,Dst Pt,Packets,Flows,Tos,numBytes,Proto_GRE,Proto_ICMP,Proto_TCP,...,Flags_.A.RS.,Flags_.A.RSF,Flags_.AP...,Flags_.AP.S.,Flags_.AP.SF,Flags_.APRS.,Flags_.APRSF,class_normal,class_suspicious,class_unknown
153613,11.819,30945,22.0,16,1,0,2227.0,0,0,1,...,0,0,0,0,1,0,0,0,1,0
33502,30.996,22400,80.0,1,1,0,46.0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
130964,11.829,22,40799.0,19,1,0,3185.0,0,0,1,...,0,0,0,0,1,0,0,0,1,0
110711,0.001,5803,23.0,1,1,0,46.0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
160562,20.162,59141,22.0,24,1,0,2900.0,0,0,1,...,0,0,0,0,1,0,0,0,1,0


In [18]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
cols_to_scale = ["Duration", "Src Pt", "Dst Pt", "Packets", "Flows", "numBytes"]
data[cols_to_scale] = scaler.fit_transform(data[cols_to_scale])

In [19]:
data.sample(5)

,Duration,Src Pt,Dst Pt,Packets,Flows,Tos,numBytes,Proto_GRE,Proto_ICMP,Proto_TCP,...,Flags_.A.RS.,Flags_.A.RSF,Flags_.AP...,Flags_.AP.S.,Flags_.AP.SF,Flags_.APRS.,Flags_.APRSF,class_normal,class_suspicious,class_unknown
108460,0.000098,0.614145,0.000336,0.000146,0.0,0,0.000351,0,0,1,...,0,0,0,0,0,0,1,0,1,0
134105,0.000019,0.000336,0.545022,0.000527,0.0,0,0.003413,0,0,1,...,0,0,0,0,1,0,0,0,1,0
63281,0.000036,0.210452,0.000336,0.000674,0.0,0,0.003107,0,0,1,...,0,0,0,0,1,0,0,0,1,0
19508,0.000000,0.078141,0.077211,0.000000,0.0,0,0.000463,0,0,0,...,0,0,0,0,0,0,0,0,1,0
18416,0.000234,0.222293,0.000336,0.000029,0.0,0,0.000135,0,0,1,...,0,0,0,1,0,0,0,0,1,0


In [20]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172838 entries, 0 to 172837
Data columns (total 39 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Duration          172838 non-null  float64
 1   Src Pt            172838 non-null  float64
 2   Dst Pt            172838 non-null  float64
 3   Packets           172838 non-null  float64
 4   Flows             172838 non-null  float64
 5   Tos               172838 non-null  int64  
 6   numBytes          172838 non-null  float64
 7   Proto_GRE         172838 non-null  int64  
 8   Proto_ICMP        172838 non-null  int64  
 9   Proto_TCP         172838 non-null  int64  
 10  Proto_UDP         172838 non-null  int64  
 11  Flags_  0x53      172838 non-null  int64  
 12  Flags_  0xc2      172838 non-null  int64  
 13  Flags_  0xd2      172838 non-null  int64  
 14  Flags_  0xd3      172838 non-null  int64  
 15  Flags_  0xd6      172838 non-null  int64  
 16  Flags_  0xd7      17

In [21]:
X = data.drop(columns=["class_normal", "class_suspicious", "class_unknown"])  # Removendo as colunas de labels
y = data[["class_normal", "class_suspicious", "class_unknown"]]

In [22]:
X.to_csv("features.csv", index=False)
y.to_csv("labels.csv", index=False)
data.to_csv("data.csv", index=False)